In [ ]:
from pathlib import Path
import sys, zipfile, numpy as np, pandas as pd

def find_data_dir():
    local_candidates = [Path.cwd() / "ai_model" / "data" / "single_sensor", Path.cwd().parent / "data" / "single_sensor"]
    for candidate in local_candidates:
        if candidate.is_dir() and list(candidate.glob("*.csv")):
            return candidate
    kaggle_root = Path("/kaggle/input")
    csv_files = list(kaggle_root.rglob("*.csv")) if kaggle_root.is_dir() else []
    if csv_files:
        return csv_files[0].parent
    zip_files = list(kaggle_root.rglob("*.zip")) if kaggle_root.is_dir() else []
    if zip_files:
        output = Path("/kaggle/working/single_sensor_dataset")
        output.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_files[0]) as archive:
            archive.extractall(output)
        extracted_csv = list(output.rglob("*.csv"))
        if extracted_csv:
            return extracted_csv[0].parent
    raise FileNotFoundError("Upload the single_sensor ZIP as a Kaggle Dataset.")

DATA_DIR = find_data_dir()
REPO_ROOT = next((candidate for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working")] if (candidate / "ai_model" / "feature_engineering.py").exists()), Path.cwd())
ROOT = REPO_ROOT
sys.path.insert(0, str(REPO_ROOT))

try:
    from ai_model.feature_engineering import load_recordings, FEATURE_COLUMNS
except ModuleNotFoundError:
    FEATURE_COLUMNS = ["distance_cm", "approach_speed_cm_s"]
    LABELS = {"SAFE": 0, "APPROACHING": 1, "DANGER": 2, "EMERGENCY": 3, "RETREATING": 0}

    def load_recordings(folder):
        frames = []
        for csv_path in sorted(Path(folder).glob("*.csv")):
            frame = pd.read_csv(csv_path)
            frame["source_file"] = csv_path.name
            frames.append(frame)
        if not frames:
            raise FileNotFoundError(f"No CSV files found in {folder}")
        frame = pd.concat(frames, ignore_index=True)
        for column in ["distance_cm", "elapsed_s"]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.dropna(subset=["distance_cm", "elapsed_s"]).copy()
        grouped = frame.groupby("source_file", sort=False)
        frame["approach_speed_cm_s"] = (grouped["distance_cm"].diff() / grouped["elapsed_s"].diff()).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        frame["label_id"] = frame["label"].astype(str).str.upper().map(LABELS).fillna(0).astype(int)
        return frame


# Single-sensor HRC Safety Log Ã¢â‚¬â€ ExtraTrees + Isolation Forest

GiÃ¡Â»Â¯ kiÃ¡ÂºÂ¿n trÃƒÂºc hybrid cÃ…Â© nhÃ†Â°ng dÃƒÂ¹ng feature mÃ¡Â»â„¢t cÃ¡ÂºÂ£m biÃ¡ÂºÂ¿n. ExtraTrees phÃƒÂ¢n loÃ¡ÂºÂ¡i nhÃƒÂ£n, Isolation Forest hÃ¡Â»Âc vÃƒÂ¹ng SAFE; hard threshold quyÃ¡ÂºÂ¿t Ã„â€˜Ã¡Â»â€¹nh EMERGENCY/WARNING trÃ†Â°Ã¡Â»â€ºc AI.

In [ ]:
from pathlib import Path
import sys, json, joblib, numpy as np, pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# load_recordings and FEATURE_COLUMNS are initialized by the Kaggle/local setup cell

In [ ]:
WARNING_CM, DANGER_CM, AI_CONFIDENCE = 60.0, 30.0, 0.70
data = load_recordings(DATA_DIR)
X, y = data[FEATURE_COLUMNS].copy(), data['label_id'].astype(int)
groups = data['source_file']
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
classifier = ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)
classifier.fit(X_train, y_train)
normal_train = X_train[y_train == 0]
anomaly_model = Pipeline([('scale', StandardScaler()), ('detector', IsolationForest(n_estimators=300, contamination=0.10, random_state=42))])
anomaly_model.fit(normal_train)
print(classification_report(y_test, classifier.predict(X_test), labels=[0,1,2,3], target_names=['SAFE','APPROACHING','DANGER','EMERGENCY'], zero_division=0))

In [ ]:
classifier_state = classifier.predict(X_test).astype(int)
classifier_confidence = classifier.predict_proba(X_test).max(axis=1)
anomaly_flag = (anomaly_model.predict(X_test) == -1).astype(int)
distance = X_test['distance_cm'].to_numpy()
states, reasons, emergency = [], [], []
for i, value in enumerate(distance):
    if value <= DANGER_CM:
        states.append('EMERGENCY'); reasons.append('HARD_DANGER_THRESHOLD'); emergency.append(True)
    elif value <= WARNING_CM:
        states.append('WARNING'); reasons.append('HARD_WARNING_THRESHOLD'); emergency.append(False)
    elif classifier_state[i] > 0 and classifier_confidence[i] >= AI_CONFIDENCE and anomaly_flag[i]:
        states.append('WARNING'); reasons.append('AI_AGREEMENT'); emergency.append(False)
    else:
        states.append('SAFE'); reasons.append('NORMAL'); emergency.append(False)
hybrid = pd.DataFrame({'state': states, 'reason': reasons, 'emergency_stop': emergency})
actual = (y_test.to_numpy() > 0).astype(int)
predicted = hybrid['state'].isin(['WARNING','EMERGENCY']).astype(int).to_numpy()
print('balanced accuracy:', balanced_accuracy_score(actual, predicted))
print('f1 hazard:', f1_score(actual, predicted, zero_division=0))
print(confusion_matrix(actual, predicted, labels=[0,1]))

In [ ]:
artifact = {'classifier': classifier, 'anomaly_model': anomaly_model, 'feature_columns': FEATURE_COLUMNS, 'thresholds_cm': {'warning': WARNING_CM, 'danger': DANGER_CM}, 'ai_confidence_threshold': AI_CONFIDENCE, 'architecture': 'hard_thresholds + ExtraTrees + IsolationForest'}
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT / 'ai_model' / 'artifacts'; output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, output_dir / 'single_sensor_extratrees_if.joblib')
print('saved:', output_dir / 'single_sensor_extratrees_if.joblib')